# Automated Classification of Coral-Reef Benthic Point Annotations
## Moorea Labeled Corals (MLC) 2010 CNN vs DNN
Ahmed Zakwaan Shareef (TP078882)

## 1. Setup & Configuration

In [ ]:
# Mount Drive: dataset zip, patch caches (.npy) and model checkpoints all live
# here so they survive Colab's 90-min idle / 12-h session limits.
import os
from google.colab import drive
drive.mount("/content/drive")
PROJECT_DRIVE = "/content/drive/MyDrive/ODL_Assignment"
os.makedirs(PROJECT_DRIVE, exist_ok=True)

In [ ]:
!pip -q install keras-tuner

In [ ]:
import re, glob, json, time, random, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt

plt.rcParams["figure.dpi"]   = 110
plt.rcParams["savefig.bbox"] = "tight"
plt.style.use("default")                     # white background for report screenshots

print("TensorFlow", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

In [ ]:
class CFG:
    # --- paths ---------------------------------------------------------
    DATASET_ZIP = f"{PROJECT_DRIVE}/Moorea_Labeled_Corals_2010.zip"   # upload this to Drive
    DATA_DIR    = "/content/Moorea Labeled Corals 2010"               # unzipped to local disk (fast I/O)

    CACHE_DIR = f"{PROJECT_DRIVE}/cache"      # extracted patch arrays (.npy)
    MODEL_DIR = f"{PROJECT_DRIVE}/models"     # checkpoints + training history + tuner trials
    FIG_DIR   = f"{PROJECT_DRIVE}/figures"    # every plot is saved here for the report

    SEED = 42

    # --- patch extraction -----------------------------------------------
    CROP_PX    = 112     # square window cut from the source image, centred on the annotated point
    INPUT_SIZE = 64      # the patch is resized to this before the model sees it
    # Zhou et al. (2024) tested 10/30/60/110 px on MLC2008 and found ~60 px best;
    # we crop wider (112) for context, then downscale.

    # --- 27 raw labels -> 6 functional groups -----------------------------
    CLASS_MAP = {
        "CCA": "CCA",
        "Sand": "Sand",
        "Turf": "Turf",
        "Macro": "Macroalgae", "Lobo": "Macroalgae",
        "Off": "Other", "Soft": "Other",
        # hard corals (Poritidae / Pocilloporidae / Acroporidae / ...)
        "Porit": "Hard coral", "P.Rus": "Hard coral", "P.Irr": "Hard coral", "Pmass": "Hard coral",
        "Pocill": "Hard coral", "Monti": "Hard coral", "Pavon": "Hard coral", "Mille": "Hard coral",
        "Acrop": "Hard coral", "Monta": "Hard coral", "Lepta": "Hard coral", "Fung": "Hard coral",
        "Gardin": "Hard coral", "Herpo": "Hard coral", "Favia": "Hard coral", "Cypha": "Hard coral",
        "Acan": "Hard coral", "Astreo": "Hard coral", "Psam": "Hard coral", "Lepto": "Hard coral",
    }
    CLASSES = ["CCA", "Hard coral", "Sand", "Turf", "Macroalgae", "Other"]

    # --- class balancing (training split only) --------------------------
    CCA_CAP = 20_000    # undersample the dominant class to at most this many training patches

    # --- splitting (by image, to prevent patch leakage) ------------------
    TEST_FRAC = 0.15
    VAL_FRAC  = 0.15    # fraction of the WHOLE dataset

    # --- training ---------------------------------------------------------
    BATCH_SIZE = 128
    EPOCHS     = 40
    LR         = 1e-3

    # --- tuning -----------------------------------------------------------
    TUNE_TRIALS = 10
    TUNE_EPOCHS = 15

for _d in (CFG.CACHE_DIR, CFG.MODEL_DIR, CFG.FIG_DIR):
    os.makedirs(_d, exist_ok=True)

N_CLASSES    = len(CFG.CLASSES)
LABEL_TO_IDX = {c: i for i, c in enumerate(CFG.CLASSES)}
IDX_TO_LABEL = {i: c for c, i in LABEL_TO_IDX.items()}

In [ ]:
def set_seed(seed=CFG.SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed()

## 2. Data Loading & Parsing

In [ ]:
# copy the zip from Drive to local disk and unzip once (reading 684 large JPEGs
# straight off a mounted Drive is slow and flaky)
if not os.path.isdir(CFG.DATA_DIR):
    assert os.path.exists(CFG.DATASET_ZIP), f"Upload the dataset zip to {CFG.DATASET_ZIP} first."
    with zipfile.ZipFile(CFG.DATASET_ZIP) as z:
        z.extractall("/content/")
    print("Unzipped ->", CFG.DATA_DIR)

assert os.path.isdir(CFG.DATA_DIR), f"DATA_DIR not found: {CFG.DATA_DIR}"
print(len(glob.glob(os.path.join(CFG.DATA_DIR, "*.jpg"))), "images found")

In [ ]:
def parse_annotations(data_dir):
    """Read every ``*.jpg.txt`` in *data_dir* into one DataFrame.

    Each file: a header line ``# Row; Col; Label`` then ~200 lines of
    ``row; col; label``. Row = pixel y (top->bottom), Col = pixel x (left->right).
    The returned DataFrame's row order IS the label<->image link: the patch array
    X and the label array y are built later in exactly this order.
    """
    records = []
    txt_files = sorted(glob.glob(os.path.join(data_dir, "*.jpg.txt")))
    assert txt_files, f"no *.jpg.txt in {data_dir}"
    for txt in txt_files:
        img_path = txt[:-4]                          # drop ".txt" -> "<name>.jpg"
        if not os.path.exists(img_path):
            print("WARNING: no image for", os.path.basename(txt)); continue
        with open(txt) as fh:
            for line in fh:
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                parts = [p.strip() for p in line.split(";")]
                if len(parts) != 3:
                    continue
                r, c, label = parts
                records.append((os.path.basename(img_path), img_path, int(r), int(c), label))
    return pd.DataFrame(records, columns=["image", "img_path", "row", "col", "label_raw"])

df = parse_annotations(CFG.DATA_DIR)
n_img = df.image.nunique()
print(f"{n_img} images | {len(df):,} annotated points | {len(df)/n_img:.0f} points/image")
df.head()

In [ ]:
# --- integrity checks -------------------------------------------------------
print("Missing values per column:\n", df.isnull().sum().to_string(), "\n")
print("Points per image:\n", df.groupby("image").size().describe().to_string(), "\n")

dims = {p: cv2.imread(p).shape[:2] for p in df.img_path.unique()}      # {path: (h, w)}
dim_df = pd.DataFrame(dims.values(), columns=["h", "w"])
print(f"Image height {dim_df.h.min()}-{dim_df.h.max()} px, "
      f"width {dim_df.w.min()}-{dim_df.w.max()} px")

df["h"] = df.img_path.map(lambda p: dims[p][0])
df["w"] = df.img_path.map(lambda p: dims[p][1])
oob = ((df.row < 0) | (df.col < 0) | (df.row >= df.h) | (df.col >= df.w)).sum()
print(f"Points on/over the border: {oob} ({100*oob/len(df):.2f}%) "
      f"-> handled by reflect-padding at extraction time")

## 3. Exploratory Data Analysis

In [ ]:
raw_counts = df.label_raw.value_counts()
print(raw_counts.to_string())

fig, ax = plt.subplots(figsize=(11, 4))
raw_counts.plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_yscale("log")
ax.set_ylabel("annotated points (log scale)")
ax.set_title(f"Raw label distribution: {len(raw_counts)} classes, "
             f"CCA = {100*raw_counts.iloc[0]/len(df):.0f}% of all points")
plt.savefig(f"{CFG.FIG_DIR}/eda_raw_class_distribution.png"); plt.show()

In [ ]:
def show_image_with_points(image_name, ax=None):
    """Full quadrat photo with its annotation points, coloured by raw label."""
    sub = df[df.image == image_name]
    img = cv2.cvtColor(cv2.imread(sub.img_path.iloc[0]), cv2.COLOR_BGR2RGB)
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(img)
    for lab, g in sub.groupby("label_raw"):
        ax.scatter(g.col, g.row, s=16, label=lab)
    ax.legend(fontsize=7, markerscale=1.4, ncol=2, loc="upper right")
    ax.set_title(image_name, fontsize=8); ax.axis("off")

show_image_with_points(sorted(df.image.unique())[0])
plt.savefig(f"{CFG.FIG_DIR}/eda_points_overlay.png"); plt.show()

In [ ]:
sample_paths = (df.img_path.drop_duplicates()
                  .sample(min(60, df.image.nunique()), random_state=CFG.SEED))
brightness = [cv2.imread(p, cv2.IMREAD_GRAYSCALE).mean() for p in sample_paths]
plt.figure(figsize=(7, 3))
plt.hist(brightness, bins=20, color="#55A868")
plt.xlabel("mean grey level"); plt.ylabel("images")
plt.title("Image brightness (60-image sample)")
plt.savefig(f"{CFG.FIG_DIR}/eda_brightness.png"); plt.show()

## 4. Preprocessing

### 4.1 Class grouping — 27 raw labels &rarr; 6 functional groups

In [ ]:
# A few files spell some labels with spaces ("P. Rus", "P mass", "P. Irr") -- normalise
# before mapping so they aren't silently dropped.
df["label_key"] = df.label_raw.str.replace(" ", "", regex=False)
df["label"]     = df.label_key.map(CFG.CLASS_MAP)

unmapped = sorted(df.loc[df.label.isna(), "label_key"].unique())
if unmapped:
    n_drop = int(df.label.isna().sum())
    print(f"Dropping {n_drop} points with labels not in CLASS_MAP: {unmapped}")
    df = df.dropna(subset=["label"]).reset_index(drop=True)

grouped_counts = df.label.value_counts().reindex(CFG.CLASSES)
print(grouped_counts.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.label_raw.value_counts().plot(kind="bar", ax=axes[0], color="#4C72B0", logy=True)
axes[0].set_title("Before: 27 raw labels"); axes[0].set_ylabel("points (log)")
grouped_counts.plot(kind="bar", ax=axes[1], color="#C44E52")
axes[1].set_title("After: 6 functional groups"); axes[1].set_ylabel("points")
plt.savefig(f"{CFG.FIG_DIR}/prep_class_grouping.png"); plt.show()

### 4.2 Patch extraction

In [ ]:
def extract_patch(img, r, c, crop=None, out=None):
    """Reflect-pad, crop a square window centred on (r, c), resize to out x out."""
    crop = crop or CFG.CROP_PX
    out  = out  or CFG.INPUT_SIZE
    half = crop // 2
    r = int(np.clip(r, 0, img.shape[0] - 1))   # clamp: a few points sit just outside the image
    c = int(np.clip(c, 0, img.shape[1] - 1))
    padded = cv2.copyMakeBorder(img, half, half, half, half, cv2.BORDER_REFLECT_101)
    rr, cc = r + half, c + half
    patch  = padded[rr - half: rr + half, cc - half: cc + half]
    return cv2.resize(patch, (out, out), interpolation=cv2.INTER_AREA)


def build_patch_arrays(frame, cache_tag):
    """Extract every patch for *frame* (row order preserved). Cached to CFG.CACHE_DIR."""
    xp = os.path.join(CFG.CACHE_DIR, f"X_{cache_tag}.npy")
    yp = os.path.join(CFG.CACHE_DIR, f"y_{cache_tag}.npy")
    gp = os.path.join(CFG.CACHE_DIR, f"groups_{cache_tag}.npy")
    if all(os.path.exists(p) for p in (xp, yp, gp)):
        print("Loading cached patches:", cache_tag)
        return np.load(xp), np.load(yp), np.load(gp, allow_pickle=True)

    n = len(frame)
    X = np.empty((n, CFG.INPUT_SIZE, CFG.INPUT_SIZE, 3), dtype=np.uint8)
    y = np.empty(n, dtype=np.int64)
    g = np.empty(n, dtype=object)
    pos, t0 = 0, time.time()
    for k, (img_path, sub) in enumerate(frame.groupby("img_path", sort=False), 1):
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        for _, rr in sub.iterrows():
            X[pos] = extract_patch(img, rr.row, rr.col)
            y[pos] = LABEL_TO_IDX[rr.label]
            g[pos] = rr.image
            pos += 1
        if k % 100 == 0:
            print(f"  {k} images, {pos:,}/{n:,} patches ({time.time()-t0:.0f}s)")
    assert pos == n
    np.save(xp, X); np.save(yp, y); np.save(gp, g)
    print(f"Extracted {n:,} patches in {time.time()-t0:.0f}s -> {xp}")
    return X, y, g


# sort by img_path and reset the index so DataFrame row k == array row k
df = df.sort_values("img_path").reset_index(drop=True)
CACHE_TAG = f"crop{CFG.CROP_PX}_in{CFG.INPUT_SIZE}"
X_all, y_all, groups_all = build_patch_arrays(df, CACHE_TAG)
print("X_all:", X_all.shape, X_all.dtype, f"({X_all.nbytes/1e9:.2f} GB in RAM)")

In [ ]:
set_seed()
fig, axes = plt.subplots(N_CLASSES, 8, figsize=(11, 1.4 * N_CLASSES))
for i, cls in enumerate(CFG.CLASSES):
    pool = np.where(y_all == LABEL_TO_IDX[cls])[0]
    for j, k in enumerate(np.random.choice(pool, 8, replace=False)):
        axes[i, j].imshow(X_all[k]); axes[i, j].axis("off")
    axes[i, 0].text(-0.35, 0.5, cls, transform=axes[i, 0].transAxes,
                    ha="right", va="center", fontsize=10)
plt.suptitle(f"Sample {CFG.INPUT_SIZE}x{CFG.INPUT_SIZE} patches per class", y=1.01)
plt.savefig(f"{CFG.FIG_DIR}/prep_sample_patches.png"); plt.show()

### 4.3 Train / validation / test split (by image, prevents patch leakage)

In [ ]:
_dummy = np.zeros(len(df))          # GroupShuffleSplit only needs shape + groups

gss1 = GroupShuffleSplit(n_splits=1, test_size=CFG.TEST_FRAC, random_state=CFG.SEED)
trainval_idx, test_idx = next(gss1.split(_dummy, y_all, groups_all))

val_rel = CFG.VAL_FRAC / (1 - CFG.TEST_FRAC)     # val fraction *within* the trainval part
gss2 = GroupShuffleSplit(n_splits=1, test_size=val_rel, random_state=CFG.SEED)
tr_rel, val_rel_idx = next(gss2.split(_dummy[trainval_idx],
                                     y_all[trainval_idx], groups_all[trainval_idx]))
train_idx = trainval_idx[tr_rel]
val_idx   = trainval_idx[val_rel_idx]

# no image may appear in more than one split
assert not (set(groups_all[train_idx]) & set(groups_all[test_idx]))
assert not (set(groups_all[val_idx])   & set(groups_all[test_idx]))
assert not (set(groups_all[train_idx]) & set(groups_all[val_idx]))

for name, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    print(f"{name:5s}  {len(idx):>7,} patches  {len(set(groups_all[idx])):>3} images")

In [ ]:
# undersample CCA in the TRAINING split only, then compute class weights
rng   = np.random.default_rng(CFG.SEED)
cca_i = LABEL_TO_IDX["CCA"]
tr_cca  = train_idx[y_all[train_idx] == cca_i]
tr_rest = train_idx[y_all[train_idx] != cca_i]
if len(tr_cca) > CFG.CCA_CAP:
    tr_cca = rng.choice(tr_cca, CFG.CCA_CAP, replace=False)
train_idx_bal = np.concatenate([tr_cca, tr_rest])
rng.shuffle(train_idx_bal)

class_weight = {i: w for i, w in enumerate(
    compute_class_weight("balanced", classes=np.arange(N_CLASSES), y=y_all[train_idx_bal]))}
print("class_weight:", {CFG.CLASSES[i]: round(w, 2) for i, w in class_weight.items()})

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, idx, title, colour in [
        (axes[0], train_idx,     "train - before undersampling", "#4C72B0"),
        (axes[1], train_idx_bal, "train - after undersampling",  "#C44E52")]:
    counts = pd.Series(y_all[idx]).value_counts().reindex(range(N_CLASSES)).fillna(0)
    counts.plot(kind="bar", ax=ax, color=colour)
    ax.set_title(title); ax.set_xticklabels(CFG.CLASSES, rotation=45, ha="right")
plt.savefig(f"{CFG.FIG_DIR}/prep_balance.png"); plt.show()

### 4.4 `tf.data` pipelines + augmentation

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def make_augment():
    """Fresh augmentation stack (new layer objects each call, so it can go in several models)."""
    return keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.15),
        layers.RandomBrightness(0.15, value_range=(0, 255)),
        layers.RandomContrast(0.15),
    ], name="augment")

def make_ds(idx, training):
    ds = tf.data.Dataset.from_tensor_slices(
        (X_all[idx], y_all[idx].astype("int32")))
    if training:
        ds = ds.shuffle(min(len(idx), 20_000), seed=CFG.SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda a, b: (tf.cast(a, tf.float32), b), num_parallel_calls=AUTOTUNE)
    return ds.batch(CFG.BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_ds(train_idx_bal, training=True)
val_ds   = make_ds(val_idx,       training=False)
test_ds  = make_ds(test_idx,      training=False)

# preview an augmented batch
_aug = make_augment()
_xb, _yb = next(iter(train_ds))
_xb_aug = _aug(_xb, training=True)
fig, axes = plt.subplots(2, 8, figsize=(11, 3))
for ax, im, lab in zip(axes.ravel(), _xb_aug, _yb):
    ax.imshow(tf.cast(tf.clip_by_value(im, 0, 255), tf.uint8)); ax.axis("off")
    ax.set_title(CFG.CLASSES[int(lab)], fontsize=7)
plt.suptitle("Augmented training batch"); plt.savefig(f"{CFG.FIG_DIR}/prep_augmented.png"); plt.show()

In [ ]:
np.savez(os.path.join(CFG.CACHE_DIR, "splits.npz"),
         train=train_idx_bal, val=val_idx, test=test_idx,
         class_weight=np.array([class_weight[i] for i in range(N_CLASSES)]))
print("saved ->", os.path.join(CFG.CACHE_DIR, "splits.npz"))

## 5. Model 1 — Convolutional Neural Network

In [ ]:
def build_cnn(input_size=CFG.INPUT_SIZE, n_classes=N_CLASSES, lr=CFG.LR):
    model = keras.Sequential([
        keras.Input((input_size, input_size, 3)),
        make_augment(),
        layers.Rescaling(1.0 / 255),

        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),

        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),

        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Dropout(0.30),

        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.50),
        layers.Dense(n_classes, activation="softmax"),
    ], name="cnn_from_scratch")

    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

cnn = build_cnn()
cnn.summary()

In [ ]:
def train_model(model, tag, train_ds, val_ds, epochs=CFG.EPOCHS, class_weight=None):
    ckpt = os.path.join(CFG.MODEL_DIR, f"{tag}.keras")
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=8,
                                      restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                          patience=4, min_lr=1e-6),
        keras.callbacks.ModelCheckpoint(ckpt, monitor="val_accuracy", save_best_only=True),
    ]
    t0 = time.time()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=epochs,
                     class_weight=class_weight, callbacks=callbacks, verbose=2)
    dt = time.time() - t0
    print(f"\n{tag}: {dt:.0f}s | best val_accuracy = {max(hist.history['val_accuracy']):.4f}")
    with open(os.path.join(CFG.MODEL_DIR, f"{tag}_history.json"), "w") as fh:
        json.dump(hist.history, fh)
    return hist

set_seed()
cnn_hist = train_model(cnn, "cnn", train_ds, val_ds, class_weight=class_weight)

In [ ]:
def plot_curves(hist, tag):
    h = hist.history if hasattr(hist, "history") else hist
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(h["accuracy"], label="train"); ax[0].plot(h["val_accuracy"], label="val")
    ax[0].set_title(f"{tag} - accuracy"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(h["loss"], label="train"); ax[1].plot(h["val_loss"], label="val")
    ax[1].set_title(f"{tag} - loss"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.savefig(f"{CFG.FIG_DIR}/{tag}_learning_curves.png"); plt.show()

plot_curves(cnn_hist, "cnn")

In [ ]:
def quick_eval(model, tag, test_ds):
    """Test-set accuracy + macro-F1 (macro-F1 matters because the test set is imbalanced)."""
    y_true = np.concatenate([y.numpy() for _, y in test_ds])
    y_pred = model.predict(test_ds, verbose=0).argmax(1)
    acc, mf1 = (y_true == y_pred).mean(), f1_score(y_true, y_pred, average="macro")
    print(f"{tag}: test accuracy = {acc:.4f} | macro-F1 = {mf1:.4f}")
    return y_true, y_pred

cnn_true, cnn_pred = quick_eval(cnn, "cnn", test_ds)

## 6. Model 2 — Deep Neural Network (fully connected)

In [ ]:
def build_dnn(input_size=CFG.INPUT_SIZE, n_classes=N_CLASSES, lr=CFG.LR):
    model = keras.Sequential([
        keras.Input((input_size, input_size, 3)),
        make_augment(),
        layers.Rescaling(1.0 / 255),
        layers.Flatten(),
        layers.Dense(512, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(n_classes, activation="softmax"),
    ], name="dnn_from_scratch")

    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

dnn = build_dnn()
dnn.summary()

In [ ]:
set_seed()
dnn_hist = train_model(dnn, "dnn", train_ds, val_ds, class_weight=class_weight)
plot_curves(dnn_hist, "dnn")
dnn_true, dnn_pred = quick_eval(dnn, "dnn", test_ds)

## 7. Hyperparameter Tuning

In [ ]:
def cnn_hypermodel(hp):
    lr    = hp.Choice("lr", [1e-2, 3e-3, 1e-3, 3e-4])
    f0    = hp.Choice("filters", [16, 32, 48])
    drop  = hp.Float("dropout", 0.2, 0.5, step=0.1)
    dense = hp.Choice("dense_units", [64, 128, 256])

    model = keras.Sequential([keras.Input((CFG.INPUT_SIZE, CFG.INPUT_SIZE, 3)),
                              make_augment(), layers.Rescaling(1.0 / 255)])
    for mult in (1, 2, 4):
        model.add(layers.Conv2D(f0 * mult, 3, padding="same", activation="relu"))
        model.add(layers.Conv2D(f0 * mult, 3, padding="same", activation="relu"))
        model.add(layers.MaxPooling2D())
        model.add(layers.Dropout(drop))
    model.add(layers.Flatten())
    model.add(layers.Dense(dense, activation="relu"))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(N_CLASSES, activation="softmax"))
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def dnn_hypermodel(hp):
    lr    = hp.Choice("lr", [1e-2, 3e-3, 1e-3, 3e-4])
    u1    = hp.Choice("units1", [256, 512, 768])
    u2    = hp.Choice("units2", [128, 256, 384])
    drop  = hp.Float("dropout", 0.2, 0.5, step=0.1)

    model = keras.Sequential([
        keras.Input((CFG.INPUT_SIZE, CFG.INPUT_SIZE, 3)),
        make_augment(), layers.Rescaling(1.0 / 255), layers.Flatten(),
        layers.Dense(u1, activation="relu"), layers.Dropout(drop),
        layers.Dense(u2, activation="relu"), layers.Dropout(drop),
        layers.Dense(N_CLASSES, activation="softmax"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

In [ ]:
cnn_tuner = kt.RandomSearch(cnn_hypermodel, objective="val_accuracy",
                            max_trials=CFG.TUNE_TRIALS, executions_per_trial=1,
                            directory=CFG.MODEL_DIR, project_name="cnn_tuning",
                            seed=CFG.SEED, overwrite=True)
cnn_tuner.search(train_ds, validation_data=val_ds, epochs=CFG.TUNE_EPOCHS,
                 class_weight=class_weight,
                 callbacks=[keras.callbacks.EarlyStopping(monitor="val_accuracy",
                                                          patience=4, restore_best_weights=True)])

dnn_tuner = kt.RandomSearch(dnn_hypermodel, objective="val_accuracy",
                            max_trials=CFG.TUNE_TRIALS, executions_per_trial=1,
                            directory=CFG.MODEL_DIR, project_name="dnn_tuning",
                            seed=CFG.SEED, overwrite=True)
dnn_tuner.search(train_ds, validation_data=val_ds, epochs=CFG.TUNE_EPOCHS,
                 class_weight=class_weight,
                 callbacks=[keras.callbacks.EarlyStopping(monitor="val_accuracy",
                                                          patience=4, restore_best_weights=True)])

In [ ]:
def plot_tuner_trials(tuner, tag):
    scores = sorted((t.score for t in tuner.oracle.trials.values() if t.score is not None),
                    reverse=True)
    plt.figure(figsize=(6, 3))
    plt.bar(range(len(scores)), scores, color="#4C72B0")
    plt.xlabel("trial (sorted)"); plt.ylabel("val_accuracy")
    plt.title(f"{tag} - tuning trials")
    plt.savefig(f"{CFG.FIG_DIR}/{tag}_tuning_trials.png"); plt.show()

plot_tuner_trials(cnn_tuner, "cnn")
plot_tuner_trials(dnn_tuner, "dnn")

In [ ]:
cnn_best_hp = cnn_tuner.get_best_hyperparameters(1)[0]
dnn_best_hp = dnn_tuner.get_best_hyperparameters(1)[0]
print("CNN best hyperparameters:", cnn_best_hp.values)
print("DNN best hyperparameters:", dnn_best_hp.values)

cnn_tuned = cnn_tuner.hypermodel.build(cnn_best_hp)
dnn_tuned = dnn_tuner.hypermodel.build(dnn_best_hp)

set_seed()
cnn_tuned_hist = train_model(cnn_tuned, "cnn_tuned", train_ds, val_ds, class_weight=class_weight)
set_seed()
dnn_tuned_hist = train_model(dnn_tuned, "dnn_tuned", train_ds, val_ds, class_weight=class_weight)

plot_curves(cnn_tuned_hist, "cnn_tuned")
plot_curves(dnn_tuned_hist, "dnn_tuned")

cnn_tuned_true, cnn_tuned_pred = quick_eval(cnn_tuned, "cnn_tuned", test_ds)
dnn_tuned_true, dnn_tuned_pred = quick_eval(dnn_tuned, "dnn_tuned", test_ds)

In [ ]:
tuning_summary = pd.DataFrame({
    "model": ["CNN - baseline", "CNN - tuned", "DNN - baseline", "DNN - tuned"],
    "test_acc": [
        (cnn_true == cnn_pred).mean(),
        (cnn_tuned_true == cnn_tuned_pred).mean(),
        (dnn_true == dnn_pred).mean(),
        (dnn_tuned_true == dnn_tuned_pred).mean(),
    ],
    "macro_f1": [
        f1_score(cnn_true, cnn_pred, average="macro"),
        f1_score(cnn_tuned_true, cnn_tuned_pred, average="macro"),
        f1_score(dnn_true, dnn_pred, average="macro"),
        f1_score(dnn_tuned_true, dnn_tuned_pred, average="macro"),
    ],
})
tuning_summary[["test_acc", "macro_f1"]] = tuning_summary[["test_acc", "macro_f1"]].round(4)
print(tuning_summary.to_string(index=False))
tuning_summary.to_csv(f"{CFG.FIG_DIR}/tuning_before_after.csv", index=False)

## 8. Evaluation

In [ ]:
def full_report(y_true, y_pred, tag):
    print(f"===== {tag} =====")
    print(classification_report(y_true, y_pred, target_names=CFG.CLASSES, digits=3))

    cm = confusion_matrix(y_true, y_pred, normalize="true")
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(CFG.CLASSES, rotation=45, ha="right")
    ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CFG.CLASSES)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center",
                    color="white" if cm[i, j] > 0.5 else "black", fontsize=8)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    ax.set_title(f"{tag} - row-normalised confusion matrix")
    fig.colorbar(im, fraction=0.046)
    plt.savefig(f"{CFG.FIG_DIR}/{tag.lower()}_confusion.png"); plt.show()

full_report(cnn_tuned_true, cnn_tuned_pred, "CNN (tuned)")
full_report(dnn_tuned_true, dnn_tuned_pred, "DNN (tuned)")

## 9. Comparison & Conclusion

In [ ]:
summary = pd.DataFrame({
    "model":    ["CNN (tuned)", "DNN (tuned)"],
    "params":   [cnn_tuned.count_params(), dnn_tuned.count_params()],
    "test_acc": [(cnn_tuned_true == cnn_tuned_pred).mean(),
                 (dnn_tuned_true == dnn_tuned_pred).mean()],
    "macro_f1": [f1_score(cnn_tuned_true, cnn_tuned_pred, average="macro"),
                 f1_score(dnn_tuned_true, dnn_tuned_pred, average="macro")],
})
summary["test_acc"] = summary.test_acc.round(4)
summary["macro_f1"] = summary.macro_f1.round(4)
print(summary.to_string(index=False))
summary.to_csv(f"{CFG.FIG_DIR}/model_comparison.csv", index=False)